# 3장. 파이토치와 자동 미분

순전파만 쓰면 역전파는 자동으로 계산되게 할 수 있는가.

**준비:** 한국전력거래소 「시간별 전국 전력수요량」 CSV 두 개(2023년, 2025년)와 기상청 종관기상관측(ASOS) 시간자료 CSV 두 개(2023년, 2025년)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 2분쯤 걸린다.

## 3.1 작은 자동 미분 프로그램 직접 만들기

In [ ]:
class Value:
    """숫자 하나와, 그 숫자를 만든 값들과 연산을 함께 기억한다."""

    def __init__(self, data, parents=()):
        self.data = data            # 순전파에서 계산한 값
        self.grad = 0.0             # 역전파에서 구할 기울기
        self.parents = parents      # 이 값을 만드는 데 쓰인 값들
        self.rule = None            # 받은 기울기를 parents로 넘기는 함수

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def rule():                 # 더하기: 받은 기울기를 그대로 넘긴다
            self.grad += out.grad
            other.grad += out.grad
        out.rule = rule
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def rule():                 # 곱하기: 상대편의 값을 곱해서 넘긴다
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out.rule = rule
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,))
        def rule():                 # ReLU: 켜져 있을 때만 넘긴다
            self.grad += out.grad * (1.0 if self.data > 0 else 0.0)
        out.rule = rule
        return out

    def backward(self):
        """이 값에서 시작해서, 계산된 순서의 반대로 rule을 실행한다."""
        order, seen = [], set()
        def visit(node):            # parents를 먼저 넣으므로 order는 계산된 순서가 된다
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node.parents:
                    visit(parent)
                order.append(node)
        visit(self)
        for node in order:
            if node.parents:        # 계산으로 만들어진 값의 기울기는 매번 새로 구한다
                node.grad = 0.0
        self.grad = 1.0
        for node in reversed(order):
            if node.rule:
                node.rule()

In [ ]:
# 2.1절의 손 계산과 같은 숫자다
x, y = Value(2.0), Value(5.0)
w, b, v, c = Value(0.5), Value(1.0), Value(1.5), Value(0.0)

out = (w * x + b).relu() * v + c       # 순전파만 적는다
diff = out + y * -1.0
L = diff * diff

L.backward()
print(out.data, L.data)
print(w.grad, b.grad, v.grad, c.grad)

In [ ]:
p = Value(3.0)
q = p * p                # p가 두 번 쓰인다
q.backward()
print(q.data, p.grad)

L.backward()             # 위에서 한 번 한 역전파를 한 번 더 한다
print(w.grad, b.grad, v.grad, c.grad)

## 3.2 텐서와 자동 미분: 2장의 기울기와 맞추기

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
h["날짜"] = (h.index - pd.Timedelta(hours=1)).normalize()
d = h.groupby("날짜").agg(GW=("GW", "mean"), 기온=("기온", "mean"))

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])
weekday = d[(d.index.dayofweek < 5) & ~d.index.isin(HOLIDAYS)]
T23, Y23 = weekday.loc["2023", "기온"].to_numpy(), weekday.loc["2023", "GW"].to_numpy()
T25, Y25 = weekday.loc["2025", "기온"].to_numpy(), weekday.loc["2025", "GW"].to_numpy()
print(len(Y23), len(Y25))

In [ ]:
def rmse(actual, predicted):
    return float(np.sqrt(((actual - predicted) ** 2).mean()))

# 1장, 2장과 같다. 표준화에 쓰는 평균과 표준편차는 2023년에서만 구한다
t_mean, t_std = T23.mean(), T23.std()
y_mean, y_std = Y23.mean(), Y23.std()
xs, ys = (T23 - t_mean) / t_std, (Y23 - y_mean) / y_std

def start(seed):
    return np.append(np.random.default_rng(seed).normal(0, 1, 6), 0.0)

def np_forward(theta, x):
    """2장의 순전파. theta = (w1, w2, b1, b2, v1, v2, c)."""
    w, b, v, c = theta[0:2], theta[2:4], theta[4:6], theta[6]
    z = np.outer(x, w) + b
    a = np.maximum(0, z)
    return a @ v + c, z, a

def np_backward(theta, x=xs, y=ys):
    """2장의 역전파. 수치 미분으로 검증한 코드다."""
    out, z, a = np_forward(theta, x)
    d_out = 2 * (out - y) / len(x)
    d_z = np.outer(d_out, theta[4:6]) * (z > 0)
    return np.concatenate([x @ d_z, d_z.sum(axis=0), a.T @ d_out, [d_out.sum()]])

def gap(g1, g2):
    return float(np.linalg.norm(g1 - g2) / (np.linalg.norm(g1) + np.linalg.norm(g2)))

theta0 = start(3)
print(np_backward(theta0).round(3).tolist())

In [ ]:
import time

def value_grad(theta):
    """Value로 247일의 평균 제곱 오차를 계산하고 역전파한다."""
    params = [Value(float(t)) for t in theta]
    w1, w2, b1, b2, v1, v2, c = params
    total = Value(0.0)
    for x_day, y_day in zip(xs, ys):
        out = (w1 * x_day + b1).relu() * v1 + (w2 * x_day + b2).relu() * v2 + c
        diff = out + (-y_day)
        total = total + diff * diff
    L = total * (1 / len(xs))
    L.backward()
    return L.data, np.array([p.grad for p in params])

began = time.perf_counter()
value_loss, by_value = value_grad(theta0)
t_value = time.perf_counter() - began
began = time.perf_counter()
by_numpy = np_backward(theta0)
t_numpy = time.perf_counter() - began
print(round(value_loss, 4), by_value.round(3).tolist())
print(f"{gap(by_value, by_numpy):.1e}")

In [ ]:
print(f"Value: {t_value * 1000:.1f}밀리초, 넘파이: {t_numpy * 1000:.3f}밀리초")

In [ ]:
import torch

t = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(t.shape, t.dtype)
print((t * 2 + 1).sum(axis=0))
print(torch.tensor([1.0, 2.0]).dtype, torch.tensor(np.array([1.0, 2.0])).dtype)
big = 16777216.0                                   # 2의 24제곱
print(torch.tensor(big) + 1, torch.tensor(big, dtype=torch.float64) + 1)

In [ ]:
x, y = torch.tensor(2.0), torch.tensor(5.0)
w = torch.tensor(0.5, requires_grad=True)      # 기울기를 구할 값이라는 표시
b = torch.tensor(1.0, requires_grad=True)
v = torch.tensor(1.5, requires_grad=True)
c = torch.tensor(0.0, requires_grad=True)

out = torch.relu(w * x + b) * v + c            # 순전파만 적는다
L = (out - y) ** 2
L.backward()
print(out.item(), L.item())
print(w.grad.item(), b.grad.item(), v.grad.item(), c.grad.item())
print(type(out.grad_fn).__name__, type(L.grad_fn).__name__)
print(x.requires_grad, w.requires_grad, out.requires_grad)

In [ ]:
try:
    L.backward()                               # 같은 L에서 역전파를 한 번 더 한다
except RuntimeError as error:
    print(str(error)[:52])

L = (torch.relu(w * x + b) * v + c - y) ** 2   # 순전파를 다시 하고 역전파한다
L.backward()
print(w.grad.item(), b.grad.item(), v.grad.item(), c.grad.item())

In [ ]:
def torch_grad(theta, dtype):
    """파이토치로 2023년 247일의 손실과 기울기를 구한다. dtype은 실수의 자료형이다."""
    t = torch.tensor(theta, dtype=dtype, requires_grad=True)
    X, Y = torch.tensor(xs, dtype=dtype), torch.tensor(ys, dtype=dtype)
    z = X[:, None] * t[0:2] + t[2:4]             # (247, 2)
    out = torch.relu(z) @ t[4:6] + t[6]          # (247,)
    L = ((out - Y) ** 2).mean()
    L.backward()
    return L.item(), t.grad.numpy().astype(np.float64)

for dtype in [torch.float64, torch.float32]:
    L_value, by_torch = torch_grad(theta0, dtype)
    print(dtype, f"{L_value:.10f}", by_torch.round(3).tolist())
    worst = np.abs(by_torch - by_numpy).max()
    print("   ", f"{worst:.1e}", f"{gap(by_torch, by_numpy):.1e}")

In [ ]:
def loss_as(theta, dtype):
    with torch.no_grad():                        # 기울기가 필요 없는 계산이라는 표시
        t = torch.tensor(theta, dtype=dtype)
        X, Y = torch.tensor(xs, dtype=dtype), torch.tensor(ys, dtype=dtype)
        out = torch.relu(X[:, None] * t[0:2] + t[2:4]) @ t[4:6] + t[6]
        return ((out - Y) ** 2).mean().item()

def numeric_as(theta, dtype, eps):
    g = np.zeros(len(theta))
    for i in range(len(theta)):
        move = np.zeros(len(theta))
        move[i] = eps
        g[i] = (loss_as(theta + move, dtype) - loss_as(theta - move, dtype)) / (2 * eps)
    return g

for eps in [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]:
    g64 = numeric_as(theta0, torch.float64, eps)
    g32 = numeric_as(theta0, torch.float32, eps)
    print(f"{eps:.0e}  {gap(g64, by_numpy):.1e}  {gap(g32, by_numpy):.1e}")

## 3.3 학습 루프 쓰기: 모델, 손실, 옵티마이저

In [ ]:
from torch import nn

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(1, 2), nn.ReLU(), nn.Linear(2, 1))
print(model)
for name, param in model.named_parameters():
    print(name, tuple(param.shape), param.dtype, param.requires_grad)
print(sum(param.numel() for param in model.parameters()))

In [ ]:
def make_model(theta):
    """2장의 theta = (w1, w2, b1, b2, v1, v2, c)를 넣은 64비트 모델."""
    model = nn.Sequential(nn.Linear(1, 2), nn.ReLU(), nn.Linear(2, 1)).double()
    with torch.no_grad():
        model[0].weight.copy_(torch.tensor(theta[0:2, None]))      # (2, 1)
        model[0].bias.copy_(torch.tensor(theta[2:4]))              # (2,)
        model[2].weight.copy_(torch.tensor(theta[None, 4:6]))      # (1, 2)
        model[2].bias.copy_(torch.tensor(theta[6:7]))              # (1,)
    return model

def theta_of(model):
    """모델의 가중치와 편향을 2장의 theta 순서로 꺼낸다."""
    parts = [model[0].weight, model[0].bias, model[2].weight, model[2].bias]
    return np.concatenate([part.detach().numpy().ravel() for part in parts])

X = torch.tensor(xs[:, None])        # (247, 1)
Y = torch.tensor(ys[:, None])        # (247, 1)
loss_fn = nn.MSELoss()
model = make_model(start(3))
print(X.shape, Y.shape, model(X).shape)
print(round(loss_fn(model(X), Y).item(), 4), theta_of(model).round(2).tolist())

In [ ]:
def fit(model, steps=5000, rate=0.05):
    optimizer = torch.optim.SGD(model.parameters(), lr=rate)
    trace = []
    for step in range(steps):
        optimizer.zero_grad()              # ① 지난 걸음에서 구한 기울기를 지운다
        loss = loss_fn(model(X), Y)        # ② 순전파로 예측과 손실을 계산한다
        loss.backward()                    # ③ 역전파로 기울기를 구한다
        optimizer.step()                   # ④ 기울기의 반대 방향으로 가중치를 고친다
        trace.append(loss.item())
    return np.array(trace)

model = make_model(start(3))
trace_right = fit(model)
print(round(trace_right[0], 4), round(trace_right[-1], 4))
print(theta_of(model).round(2).tolist())

theta = start(3)                           # 2장의 넘파이 경사하강법
for _ in range(5000):
    theta = theta - 0.05 * np_backward(theta)
print(f"{np.abs(theta_of(model) - theta).max():.1e}")

In [ ]:
def gw(model, t):
    """기온(°C)을 받아 전력수요(GW)를 낸다. 평가할 때는 기울기를 기록하지 않는다."""
    scaled = torch.tensor(((t - t_mean) / t_std)[:, None], dtype=model[0].weight.dtype)
    with torch.no_grad():
        out = model(scaled)
    return out.numpy()[:, 0].astype(np.float64) * y_std + y_mean

with torch.no_grad():
    quiet = model(X)
print(model(X).requires_grad, quiet.requires_grad)
print(f"{rmse(Y23, gw(model, T23)):.2f}", f"{rmse(Y25, gw(model, T25)):.2f}")

In [ ]:
model = make_model(start(3))
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
print(model[2].bias.grad)              # 역전파를 하기 전
loss_fn(model(X), Y).backward()
print(model[2].bias.grad)              # 한 번 한 뒤
loss_fn(model(X), Y).backward()
print(model[2].bias.grad)              # 지우지 않고 한 번 더 한 뒤
optimizer.zero_grad()
print(model[2].bias.grad)              # 지운 뒤

In [ ]:
def fit_wrong(model, kind, steps=5000, rate=0.05):
    """학습 루프에서 한 군데를 틀리게 쓴 것. kind가 틀린 곳이다."""
    optimizer = torch.optim.SGD(model.parameters(), lr=rate)
    Y_flat = Y[:, 0]                                   # (247,)
    trace = []
    for step in range(steps):
        if kind != "기울기를 지우지 않음":
            optimizer.zero_grad()
        if kind == "모양이 다른 손실":
            loss = ((model(X) - Y_flat) ** 2).mean()   # (247, 1)에서 (247,)을 뺀다
        else:
            loss = loss_fn(model(X), Y)
        loss.backward()
        optimizer.step()
        trace.append(loss.item())
    return np.array(trace)

model = make_model(start(3))
trace_nozero = fit_wrong(model, "기울기를 지우지 않음")
print([round(float(trace_nozero[k]), 4) for k in (0, 1, 2, 3, 5, 10, 100, 1000, 4999)])
print([round(float(trace_right[k]), 4) for k in (0, 1, 2, 3)])
late = trace_nozero[1000:]
print(round(float(trace_nozero.min()), 4), int(trace_nozero.argmin()))
print(round(float(late.min()), 4), round(float(late.max()), 4))
print(int(np.where(trace_nozero < 1.0)[0].max()))
z_end = model[0](X)
print(theta_of(model).round(2).tolist(), (z_end > 0).sum(dim=0).tolist())
print(f"{rmse(Y25, gw(model, T25)):.2f}")

In [ ]:
model = make_model(start(3))
pred, Y_flat = model(X), Y[:, 0]
print(pred.shape, Y_flat.shape, (pred - Y_flat).shape)
print(round(((pred - Y_flat) ** 2).mean().item(), 4), round(loss_fn(pred, Y).item(), 4))
spread, center = pred.var(correction=0).item(), pred.mean().item()
print(round(spread, 4), round(center ** 2, 4), round(spread + 1 + center ** 2, 4))

trace_shape = fit_wrong(model, "모양이 다른 손실")
print(round(float(trace_shape[0]), 4), round(float(trace_shape[-1]), 4))
print((theta_of(model).round(2) + 0.0).tolist(), round(rmse(Y25, gw(model, T25)), 2))

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    loss_fn(make_model(start(3))(X), Y_flat)
print(len(caught), str(caught[0].message)[:80])

## 3.4 Dataset과 DataLoader로 미니배치 만들기

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X, Y)
shuffler = torch.Generator().manual_seed(0)       # 섞는 순서를 정하는 난수 생성기
loader = DataLoader(dataset, batch_size=32, shuffle=True, generator=shuffler)
print(len(dataset), len(loader))
for epoch in range(2):
    sizes, first = [], None
    for x_batch, y_batch in loader:
        sizes.append(len(x_batch))
        if first is None:
            first = x_batch[:3, 0] * t_std + t_mean   # 첫 미니배치 앞 세 날의 기온(°C)
    print(epoch, sizes, first.numpy().round(1).tolist())

In [ ]:
def fit_batches(model, epochs, batch_size=32, seed=0, rate=0.05):
    """미니배치 경사하강법. 미니배치 하나마다 가중치를 한 번 고친다."""
    shuffler = torch.Generator().manual_seed(seed)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        generator=shuffler)
    optimizer = torch.optim.SGD(model.parameters(), lr=rate)
    per_step, per_epoch = [], []
    for epoch in range(epochs):
        for x_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
            per_step.append(loss.item())
        with torch.no_grad():
            per_epoch.append(loss_fn(model(X), Y).item())     # 247일 전체의 손실
    return np.array(per_step), np.array(per_epoch)

model = make_model(start(3))
step_trace, epoch_trace = fit_batches(model, epochs=625)
print(len(step_trace), [round(float(epoch_trace[k]), 4) for k in (0, 9, 99, 624)])
print([round(float(trace_right[k]), 4) for k in (7, 79, 799, 4999)])
print(int(np.argmax(epoch_trace < 0.23)) + 1, int(np.argmax(trace_right < 0.23)) + 1)
last = epoch_trace[-10:]
print(round(float(last.min()), 4), round(float(last.max()), 4))
print(round(float(step_trace[-8:].min()), 4), round(float(step_trace[-8:].max()), 4))
print(round(rmse(Y23, gw(model, T23)), 2), round(rmse(Y25, gw(model, T25)), 2))

In [ ]:
whole, small = [], []
for seed in range(10):
    model = make_model(start(seed))
    fit(model)
    whole.append(round(rmse(Y25, gw(model, T25)), 2))
    model = make_model(start(seed))
    fit_batches(model, epochs=625, seed=seed)
    small.append(round(rmse(Y25, gw(model, T25)), 2))
print(whole, sum(mark < 3.5 for mark in whole))
print(small, sum(mark < 3.5 for mark in small))

## 3.5 같은 결과를 다시 얻기: 시드, 자료형, 스레드

In [ ]:
def run(seed=None, extra=False, dtype=torch.float64):
    """유닛 8개인 신경망을 기본 초기화로 만들어 미니배치로 200에폭 학습시킨다."""
    if seed is not None:
        torch.manual_seed(seed)
    if extra:
        nn.Linear(3, 3)                     # 쓰지 않는 층을 하나 만든다
    model = nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1)).to(dtype)
    data = TensorDataset(X.to(dtype), Y.to(dtype))
    loader = DataLoader(data, batch_size=32, shuffle=True)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
    for epoch in range(200):
        for x_batch, y_batch in loader:
            optimizer.zero_grad()
            loss_fn(model(x_batch), y_batch).backward()
            optimizer.step()
    return rmse(Y25, gw(model, T25))

print("같은 시드로 두 번:", f"{run(0):.8f}", f"{run(0):.8f}")
torch.manual_seed(0)
print("시드를 한 번만 정하고 두 번:", f"{run():.8f}", f"{run():.8f}")
print("모델 앞에 층을 하나 더 만들고:", f"{run(0, extra=True):.8f}")
print("32비트 실수로:", f"{run(0, dtype=torch.float32):.8f}")

In [ ]:
before = torch.get_num_threads()
torch.set_num_threads(1)
one = run(0)
torch.set_num_threads(2)
two = run(0)
torch.set_num_threads(before)
print(before, f"{one:.8f}", f"{two:.8f}")
print([round(run(seed), 2) for seed in range(10)])

In [ ]:
import sys, platform

print(sys.version.split()[0], np.__version__, torch.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.cuda.is_available())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 3-3
sweep = np.logspace(-8, 0, 33)
e64 = np.array([gap(numeric_as(theta0, torch.float64, e), by_numpy) for e in sweep])
e32 = np.array([gap(numeric_as(theta0, torch.float32, e), by_numpy) for e in sweep])
auto32 = gap(torch_grad(theta0, torch.float32)[1], by_numpy)
auto64 = gap(torch_grad(theta0, torch.float64)[1], by_numpy)
sci = FuncFormatter(lambda val, _: f"{val:.0e}")
fig, ax = plt.subplots(figsize=(7, 5.0))
ax.plot(sweep, np.maximum(e64, 1e-17), color=NAVY, lw=1.8, marker="o", ms=3.2, label="수치 미분, 64비트 실수")
ax.plot(sweep, np.minimum(e32, 1), color=BRICK, lw=1.8, marker="o", ms=3.2, label="수치 미분, 32비트 실수")
ax.axhline(auto32, color=TEAL, lw=1.6, ls=(0, (5, 2)), label=f"자동 미분, 32비트 실수: {auto32:.1e}")
ax.axhline(auto64, color=TEAL, lw=1.6, label=f"자동 미분, 64비트 실수: {auto64:.1e}")
k32 = int(e32.argmin())
ax.scatter([sweep[k32]], [e32[k32]], s=46, color=BRICK, zorder=5, edgecolor="white", lw=0.8)
ax.text(7e-2, 6e-6, f"32비트에서 가장 작은 오차\nε = {sweep[k32]:.0e}, {e32[k32]:.0e}", fontsize=9, color=BRICK, ha="left", va="center", linespacing=1.5)
i5 = int(np.abs(np.log10(sweep) + 5).argmin())
ax.scatter([sweep[i5], sweep[i5]], [e64[i5], e32[i5]], s=40, color=[NAVY, BRICK], zorder=5, edgecolor="white", lw=0.8)
ax.text(1e-5 * 1.5, e32[i5] * 2.2, f"ε = 1e-05, 32비트: {e32[i5]:.0e}", fontsize=9, color=BRICK, ha="left", va="bottom")
ax.text(1e-5 * 0.7, e64[i5] * 0.35, f"ε = 1e-05, 64비트: {e64[i5]:.0e}", fontsize=9, color=NAVY, ha="right", va="top")
ax.axhline(0.04, color=GRAY, lw=1.0, ls=(0, (2, 2)))
ax.text(1.6, 0.04 * 1.6, "2장의 틀린 코드에서 가장 작았던 오차 0.04", fontsize=9, color="#5b6470", ha="right", va="bottom")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(5e-9, 2); ax.set_ylim(1e-17, 3)
ax.set_xticks([1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1])
ax.set_yticks([1e-16, 1e-14, 1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 1])
ax.xaxis.set_major_formatter(sci); ax.yaxis.set_major_formatter(sci)
ax.minorticks_off()
ax.set_xlabel("수치 미분의 간격 ε (로그 축)", fontsize=9)
ax.set_ylabel("2장의 역전파와의 상대 오차 (로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(0.0, 0.76))
frame(ax)
fig.tight_layout()
fig.savefig("fig_3_3.png", dpi=300)
plt.show()

In [ ]:
# 그림 3-5
steps = np.arange(1, 5001)
fig, axes = plt.subplots(2, 1, figsize=(7, 7.6))
for ax, (lo, hi) in zip(axes, [(1, 50), (1, 5000)]):
    ax.plot(steps, trace_nozero, color=BRICK, lw=1.3, label=f"기울기를 지우지 않음: 5,000걸음 뒤 {trace_nozero[-1]:.4f}")
    ax.plot(steps, trace_shape, color=NAVY, lw=1.6, ls=(0, (5, 2)), label=f"모양이 다른 손실: 5,000걸음 뒤 {trace_shape[-1]:.4f}")
    ax.plot(steps, trace_right, color=TEAL, lw=2.4, label=f"올바른 학습 루프: 5,000걸음 뒤 {trace_right[-1]:.4f}")
    ax.set_xlim(lo, hi); ax.set_ylim(0, 2.4)
    ax.set_ylabel("그 걸음에서 계산한 손실", fontsize=9)
    frame(ax, "① 처음 50걸음" if hi == 50 else "② 5,000걸음 전체")
axes[0].set_xticks([1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50])
axes[0].set_xlabel("가중치를 고친 횟수 (처음 50걸음)", fontsize=9)
axes[1].set_xscale("log"); plain_ticks(axes[1].xaxis); axes[1].minorticks_off()
axes[1].set_xlabel("가중치를 고친 횟수 (5,000걸음 전체, 로그 축)", fontsize=9)
axes[0].legend(frameon=False, fontsize=9, loc="upper right")
fig.tight_layout(h_pad=2.2)
fig.savefig("fig_3_5.png", dpi=300)
plt.show()

In [ ]:
# 그림 3-7
grid_t = np.linspace(-10, 31, 411)
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.scatter(T23, Y23, s=10, color=to_rgba(GRAY, 0.6), edgecolor="none")
for kind, col, ls, lw in [("기울기를 지우지 않음", BRICK, "-", 1.5), ("모양이 다른 손실", NAVY, (0, (5, 2)), 1.6),
                          ("올바른 학습 루프", TEAL, "-", 2.4)]:
    model = make_model(start(3))
    fit(model) if kind == "올바른 학습 루프" else fit_wrong(model, kind)
    mark = rmse(Y25, gw(model, T25))
    ax.plot(grid_t, gw(model, grid_t), color=col, ls=ls, lw=lw, label=f"{kind}: 2025년 RMSE {mark:.2f}GW")
ax.set_xlim(-10.5, 31.5); ax.set_ylim(54, 94); ax.set_xticks(range(-10, 31, 5))
ax.set_xlabel("일평균 기온(°C)", fontsize=9)
ax.set_ylabel("일평균 전력수요(GW)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper center")
frame(ax)
fig.tight_layout()
fig.savefig("fig_3_7.png", dpi=300)
plt.show()

In [ ]:
# 그림 3-9
updates = np.arange(1, 5001)
at_epoch = np.arange(1, 626) * 8
fig, axes = plt.subplots(2, 1, figsize=(7, 7.8))
ax = axes[0]
ax.plot(updates, step_trace, color=to_rgba(BRICK, 0.55), lw=0.6, label="미니배치 32일의 손실 (걸음마다)")
ax.plot(at_epoch, epoch_trace, color=BRICK, lw=1.8, label="미니배치 학습: 247일 전체의 손실 (에폭마다)")
ax.plot(updates, trace_right, color=TEAL, lw=2.2, label="전체 배치 학습: 247일 전체의 손실")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(1, 5000); ax.set_ylim(0.1, 3)
plain_ticks(ax.xaxis); ax.set_yticks([0.1, 0.2, 0.3, 0.5, 1, 2])
ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:g}")); ax.minorticks_off()
ax.set_xlabel("가중치를 고친 횟수 (로그 축)", fontsize=9)
ax.set_ylabel("손실 (로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper right")
frame(ax, "① 가중치를 고친 횟수로 본 손실")
ax = axes[1]
ax.plot(at_epoch / 8, epoch_trace, color=BRICK, lw=1.8, label="미니배치 학습 (한 에폭에 8번 고친다)")
ax.plot(updates, trace_right, color=TEAL, lw=2.2, label="전체 배치 학습 (한 에폭에 1번 고친다)")
ax.axhline(0.23, color=GRAY, lw=0.9, ls=(0, (2, 2)))
ax.text(4600, 0.236, "손실 0.23", fontsize=9, color="#5b6470", ha="right", va="bottom")
for trace, col in [(epoch_trace, BRICK), (trace_right, TEAL)]:
    hit = int(np.argmax(np.asarray(trace) < 0.23)) + 1          # 손실이 처음 0.23 아래로 내려간 에폭
    ax.scatter([hit], [trace[hit - 1]], s=40, color=col, edgecolor="white", lw=0.8, zorder=5)
    ax.text(hit, 0.205, f"{hit}에폭", fontsize=9, color=col, ha="center", va="top", weight="bold")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(1, 5000); ax.set_ylim(0.2, 3)
plain_ticks(ax.xaxis); ax.set_yticks([0.2, 0.3, 0.5, 1, 2])
ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:g}")); ax.minorticks_off()
ax.set_xlabel("자료 전체를 본 횟수, 에폭 (로그 축)", fontsize=9)
ax.set_ylabel("247일 전체의 손실 (로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper right")
frame(ax, "② 자료 전체를 본 횟수로 본 손실"); ax.set_ylim(0.16, 3)
fig.tight_layout(h_pad=2.2)
fig.savefig("fig_3_9.png", dpi=300)
plt.show()